In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Create directory for saving plots if it doesn't exist
save_dir = "power_and_timing"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# Load the data
data = {
    'Time (ns)': [8, 9, 10, 11, 13, 15, 25, 50, 100],
    'WNS': [-0.297, 0.014, 0.069, 0.578, 2.469, 3.613, 10.146, 22.779, 47.738],
    'TNS': [-2.230, 0.00, 0.000, 0, 0, 0, 0, 0, 0],
    'Density (%)': [59.763, 60.183, 52.364, 54.122, 53.665, 52.971, 52.665, 52.784, 52.857],
    'Internal Power (mW)': [2.177, 1.975, 1.73, 1.588, 1.301, 1.1418, 0.6848, 0.3528, 0.1705],
    'Switching Power (mW)': [0.6104, 0.535, 0.4489, 0.3903, 0.3335, 0.2851, 0.1725, 0.08557, 0.04356],
    'Total Power (mW)': [2.7874, 2.51, 2.1789, 1.9783, 1.6345, 1.4269, 0.8573, 0.43837, 0.21406],
    'Leakage power (mW)': [0.003296, 0.003123, 0.002535, 0.002511, 0.002483, 0.00245, 0.002436, 0.002427, 0.002431],
    'Total Capacitance (10^-11 F)': [3.265, 3.183, 2.788, 2.687, 2.705, 2.69, 2.687, 2.657, 2.697]
}

df = pd.DataFrame(data)

# Calculate frequency in MHz from period in ns
df['Frequency (MHz)'] = 1000 / df['Time (ns)']

# Common figure parameters
fig_params = {'figsize': (16, 8), 'dpi': 1000}

# Plot 1: Power vs Frequency
plt.figure(**fig_params)
plt.plot(df['Frequency (MHz)'], df['Total Power (mW)'], 'o-', color='blue', linewidth=2, label='Total Power')
plt.plot(df['Frequency (MHz)'], df['Internal Power (mW)'], 's-', color='green', linewidth=2, label='Internal Power')
plt.plot(df['Frequency (MHz)'], df['Switching Power (mW)'], '^-', color='orange', linewidth=2, label='Switching Power')
plt.plot(df['Frequency (MHz)'], df['Leakage power (mW)'], 'D-', color='red', linewidth=2, label='Leakage Power')
plt.xlabel('Frequency (MHz)', fontsize=12)
plt.ylabel('Power (mW)', fontsize=12)
plt.title('Power vs Frequency', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.xlim(0, 130)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'plot1_power_vs_frequency.png'))
plt.close()

# Plot 2: Timing Slack vs Frequency
plt.figure(**fig_params)
plt.plot(df['Frequency (MHz)'], df['WNS'], 'o-', color='blue', linewidth=2, label='WNS')
plt.axhline(y=0, color='r', linestyle='--', alpha=0.7)
plt.xlabel('Frequency (MHz)', fontsize=12)
plt.ylabel('Worst Negative Slack (ns)', fontsize=12)
plt.title('Timing Slack vs Frequency', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xlim(0, 130)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'plot2_timing_vs_frequency.png'))
plt.close()

# Plot 3: Power Breakdown - stacked area chart
plt.figure(**fig_params)
x = df['Frequency (MHz)']
y1 = df['Internal Power (mW)']
y2 = df['Switching Power (mW)']
y3 = df['Leakage power (mW)']

plt.fill_between(x, 0, y3, color='red', alpha=0.7, label='Leakage Power')
plt.fill_between(x, y3, y3+y2, color='orange', alpha=0.7, label='Switching Power')
plt.fill_between(x, y3+y2, y3+y2+y1, color='green', alpha=0.7, label='Internal Power')
plt.xlabel('Frequency (MHz)', fontsize=12)
plt.ylabel('Power (mW)', fontsize=12)
plt.title('Power Breakdown by Component', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.xlim(0, 130)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'plot3_power_breakdown.png'))
plt.close()

# Plot 5: Density and Capacitance vs Frequency
fig, ax1 = plt.subplots(**fig_params)

# Plot Density on primary y-axis
ax1.plot(df['Frequency (MHz)'], df['Density (%)'], 'o-', color='blue', linewidth=2, label='Density')
ax1.set_xlabel('Frequency (MHz)', fontsize=12)
ax1.set_ylabel('Density (%)', fontsize=12, color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_xlim(0, 130)

# Create a second y-axis for capacitance
ax2 = ax1.twinx()
ax2.plot(df['Frequency (MHz)'], df['Total Capacitance (10^-11 F)'], 's-', color='green', linewidth=2, label='Capacitance')
ax2.set_ylabel('Total Capacitance (10^-11 F)', fontsize=12, color='green')
ax2.tick_params(axis='y', labelcolor='green')

# Add title and grid
plt.title('Density and Capacitance vs Frequency', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Add combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'plot5_density_capacitance.png'))
plt.close()

# Create a table showing key operating points - without efficiency
fig, ax = plt.subplots(figsize=(16, 6), dpi=1000)  # Adjusted height for better table display
operating_points = [8, 9, 10, 15, 25, 50, 100]
# Removed Power Efficiency from the table columns
table_data = df[df['Time (ns)'].isin(operating_points)][['Time (ns)', 'Frequency (MHz)', 'WNS', 'Total Capacitance (10^-11 F)', 'Total Power (mW)']]
table_data = table_data.round(3)

table = plt.table(cellText=table_data.values,
                 colLabels=table_data.columns,
                 cellLoc='center',
                 loc='center',
                 bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.5)
ax.axis('off')
plt.title('Key Operating Points Summary', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'table_operating_points.png'))
plt.close()

print(f"Selected plots have been saved to the '{save_dir}' folder with size (16, 8) and DPI=1000")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
